In [8]:
##Code for modeling 

import pandas as pd 

dev = pd.read_csv("../data/processed/development_v1.csv")
eval = pd.read_csv("../data/processed/evaluation_v1.csv")

print(dev.shape, eval.shape)
dev.head()
dev["text"] = dev["text"].astype(str).fillna("")
eval["text"] = eval["text"].astype(str).fillna("")

(79997, 10) (20000, 9)


In [9]:
dev.columns

Index(['Id', 'text', 'source', 'title', 'n_tokens', 'title_ratio', 'year',
       'month', 'has_timestamp', 'label'],
      dtype='object')

In [10]:
X_text = dev["text"]
y = dev["label"]

In [11]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

pipeline = Pipeline([
	("tfidf", TfidfVectorizer(
		max_features=30000,
		min_df=5,
		stop_words="english"
	)),
	("clf", LogisticRegression(
		max_iter=1000,
		n_jobs=-1,
		class_weight="balanced"
	))
])


In [12]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(
	n_splits=5,
	shuffle=True,
	random_state=42
)

scores = cross_val_score(
	pipeline,
	X_text,
	y,
	cv=cv,
	scoring="f1_macro",
	n_jobs=-1
)

print("Macro-F1 scores:", scores)
print("Mean Macro-F1:", scores.mean())
print("Std:", scores.std())



Macro-F1 scores: [0.62712923 0.62862841 0.63068727 0.63152104 0.63108878]
Mean Macro-F1: 0.6298109444059807
Std: 0.0016682436114110702


In [13]:
from sklearn.svm import LinearSVC

pipeline_svm = Pipeline([
	("tfidf", TfidfVectorizer(
		max_features=50000,
		min_df=5,
		stop_words="english",
		ngram_range=(1,2)
	)),
	("clf", LinearSVC(
		class_weight="balanced"
	))
])


In [14]:
scores = cross_val_score(
	pipeline_svm,
	dev["text"],
	dev["label"],
	cv=cv,
	scoring="f1_macro",
	n_jobs=-1
)

scores.mean()

np.float64(0.6254996473878318)